# Duplicate Analysis and Data Cleaning

This notebook investigates and removes duplicate records from the Instagram engagement dataset more thoroughly than the basic script.

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [ ]:
# Load the dataset
input_file = 'final_combined_dataset.csv'
df = pd.read_csv(input_file)

print(f"Original dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nTotal duplicates in original data: {df.duplicated().sum()}")

# Check memory usage
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Let's examine the duplicates more closely
print("=== Duplicate Analysis ===")

# Check duplicates based on post_id (which should be unique)
post_id_duplicates = df['post_id'].duplicated().sum()
print(f"Duplicate post_ids: {post_id_duplicates}")

# Show some duplicate post_ids
duplicate_post_ids = df[df['post_id'].duplicated(keep=False)]['post_id'].unique()
print(f"\nSample duplicate post_ids: {duplicate_post_ids[:5]}")

# Look at one specific duplicate
if len(duplicate_post_ids) > 0:
    sample_id = duplicate_post_ids[0]
    sample_duplicates = df[df['post_id'] == sample_id]
    print(f"\nExample of duplicates for post_id {sample_id}:")
    print(f"Number of records: {len(sample_duplicates)}")
    print("\nColumns that differ between duplicates:")
    
    for col in df.columns:
        unique_vals = sample_duplicates[col].nunique()
        if unique_vals > 1:
            print(f"  {col}: {unique_vals} unique values")
            print(f"    Values: {list(sample_duplicates[col].unique())}")

In [ ]:
# Features we want to keep (same as in the cleanup script)
features_to_keep = [
    'caption',
    'hashtags', 
    'media_type',
    'caption_length',
    'num_hashtags',
    'has_mention',
    'has_url',
    'has_emoji',
    'timestamp',
    'sentiment_weighted_engagement'
]

print(f"Features to keep: {len(features_to_keep)}")
print(features_to_keep)

In [ ]:
# Filter to only the columns we want
filtered_df = df[features_to_keep].copy()

print(f"Filtered dataset shape: {filtered_df.shape}")
print(f"Duplicates in filtered data: {filtered_df.duplicated().sum()}")

# Let's see what the duplicated rows look like
if filtered_df.duplicated().sum() > 0:
    print("\n=== Sample of duplicate rows in filtered data ===")
    duplicated_rows = filtered_df[filtered_df.duplicated(keep=False)]
    print(f"Total rows that are duplicates: {len(duplicated_rows)}")
    
    # Show first few duplicates
    print("\nFirst 4 duplicate rows:")
    print(duplicated_rows.head(4))
    
    # Check if they are exactly identical
    print("\n=== Checking if duplicates are exactly identical ===")
    first_dup = duplicated_rows.iloc[0]
    second_dup = duplicated_rows.iloc[1] 
    
    print("Are first two duplicate rows exactly equal?")
    for col in features_to_keep:
        val1 = first_dup[col]
        val2 = second_dup[col]
        are_equal = pd.isna(val1) and pd.isna(val2) or val1 == val2
        print(f"  {col}: {are_equal} ({repr(val1)} vs {repr(val2)})")

In [ ]:
# Strategy 1: Basic drop_duplicates (what the script currently does)
print("=== Strategy 1: Basic drop_duplicates ===")
strategy1_df = filtered_df.drop_duplicates()
print(f"Records after basic drop_duplicates: {len(strategy1_df)}")
print(f"Duplicates removed: {len(filtered_df) - len(strategy1_df)}")

# Strategy 2: More aggressive - drop duplicates but keep='first' explicitly
print("\n=== Strategy 2: drop_duplicates with keep='first' ===")
strategy2_df = filtered_df.drop_duplicates(keep='first')
print(f"Records after drop_duplicates(keep='first'): {len(strategy2_df)}")
print(f"Duplicates removed: {len(filtered_df) - len(strategy2_df)}")

# Strategy 3: Drop based on specific key columns (if timestamp + caption is enough to identify unique posts)
print("\n=== Strategy 3: Drop duplicates based on key columns ===")
key_columns = ['timestamp', 'caption', 'sentiment_weighted_engagement']
strategy3_df = filtered_df.drop_duplicates(subset=key_columns, keep='first')
print(f"Records after dropping duplicates on {key_columns}: {len(strategy3_df)}")
print(f"Duplicates removed: {len(filtered_df) - len(strategy3_df)}")

# Check if any strategy gives us truly unique records
print(f"\n=== Final duplicate check ===")
print(f"Strategy 1 remaining duplicates: {strategy1_df.duplicated().sum()}")
print(f"Strategy 2 remaining duplicates: {strategy2_df.duplicated().sum()}")
print(f"Strategy 3 remaining duplicates: {strategy3_df.duplicated().sum()}")

In [ ]:
# If duplicates still remain, let's investigate why
final_df = strategy1_df  # Use the basic strategy for now

if final_df.duplicated().sum() > 0:
    print("=== Investigating remaining duplicates ===")
    remaining_dups = final_df[final_df.duplicated(keep=False)]
    print(f"Remaining duplicate records: {len(remaining_dups)}")
    
    # Show the remaining duplicates
    print("\nRemaining duplicate records:")
    print(remaining_dups)
    
    # Check for floating point precision issues
    print("\n=== Checking for floating point precision issues ===")
    numeric_cols = final_df.select_dtypes(include=[np.number]).columns
    print(f"Numeric columns: {list(numeric_cols)}")
    
    for col in numeric_cols:
        if col in remaining_dups.columns:
            unique_vals = remaining_dups[col].nunique()
            if unique_vals > 1:
                print(f"\n{col} has {unique_vals} unique values in duplicates:")
                print(remaining_dups[col].unique())
                
                # Check if rounding helps
                rounded_vals = remaining_dups[col].round(6).nunique()
                print(f"After rounding to 6 decimals: {rounded_vals} unique values")
else:
    print("✅ No duplicates remaining after basic drop_duplicates!")

In [ ]:
# Apply the most effective duplicate removal strategy
print("=== Final Cleaning Process ===")

# Start with the filtered data
cleaned_df = filtered_df.copy()
initial_count = len(cleaned_df)
print(f"Starting with {initial_count} records")

# Method 1: Basic drop_duplicates
cleaned_df = cleaned_df.drop_duplicates()
after_basic = len(cleaned_df)
print(f"After basic drop_duplicates: {after_basic} records")

# Method 2: If there are still duplicates, try with rounded numeric values
if cleaned_df.duplicated().sum() > 0:
    print(f"Still {cleaned_df.duplicated().sum()} duplicates remaining")
    
    # Round numeric columns to handle floating point precision issues
    numeric_cols = cleaned_df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        cleaned_df[col] = cleaned_df[col].round(10)
    
    # Try again
    cleaned_df = cleaned_df.drop_duplicates()
    after_rounding = len(cleaned_df)
    print(f"After rounding and drop_duplicates: {after_rounding} records")

# Method 3: If STILL duplicates, be more aggressive
if cleaned_df.duplicated().sum() > 0:
    print(f"Still {cleaned_df.duplicated().sum()} duplicates remaining")
    
    # Force remove by resetting index and using groupby
    cleaned_df = cleaned_df.reset_index(drop=True)
    
    # Group by all columns and take first
    group_cols = list(cleaned_df.columns)
    cleaned_df = cleaned_df.groupby(group_cols, dropna=False).first().reset_index()
    
    after_groupby = len(cleaned_df)
    print(f"After groupby approach: {after_groupby} records")

final_count = len(cleaned_df)
duplicates_removed = initial_count - final_count

print(f"\n=== Final Results ===")
print(f"Original records: {initial_count}")
print(f"Final records: {final_count}")
print(f"Duplicates removed: {duplicates_removed}")
print(f"Remaining duplicates: {cleaned_df.duplicated().sum()}")

In [ ]:
# Save the cleaned dataset
output_file = 'notebook-cleaned_dataset.csv'
cleaned_df.to_csv(output_file, index=False)

print(f"\n=== Dataset Saved ===")
print(f"Cleaned dataset saved to '{output_file}'")
print(f"Final shape: {cleaned_df.shape}")
print(f"Columns: {list(cleaned_df.columns)}")

# Verify the saved file
saved_df = pd.read_csv(output_file)
print(f"\nVerification - loaded dataset shape: {saved_df.shape}")
print(f"Verification - duplicates in saved file: {saved_df.duplicated().sum()}")

In [ ]:
# Compare with the original script results
print("=== Comparison with Original Script ===")
print(f"Original script result: 537 records (removed 3,835 duplicates from 4,372)")
print(f"Notebook result: {final_count} records (removed {duplicates_removed} duplicates from {initial_count})")

if final_count != 537:
    print(f"\n⚠️  Different results! Difference: {final_count - 537} records")
    
    # Check if it's because we're more thorough
    original_cleaned = pd.read_csv('test-cleaned_dataset.csv')
    print(f"Original cleaned file duplicates: {original_cleaned.duplicated().sum()}")
    print(f"New cleaned file duplicates: {saved_df.duplicated().sum()}")
else:
    print("\n✅ Same result as original script!")

print(f"\n=== Final Summary ===")
print(f"✅ Successfully cleaned dataset")
print(f"✅ Removed all {duplicates_removed} duplicate records")
print(f"✅ Final dataset: {final_count} unique records with {len(features_to_keep)} features")
print(f"✅ Zero duplicates remaining: {cleaned_df.duplicated().sum() == 0}")